In [1]:
import sys
import os

# Adds your project root directory to the python module search path
sys.path.append(os.path.abspath(os.path.join('..')))

# Now change your import statement to use the folder path name
import mysql.connector
import pandas as pd
from app.db_functions import connect_to_db

# Test the connection string
try:
    db = connect_to_db()
    cursor = db.cursor(dictionary=True)
    cursor.execute("SHOW TABLES;")
    print("Kernel hooked up perfectly! Found tables:", cursor.fetchall())
    cursor.close()
    db.close()
except Exception as e:
    print(f"Connection check failed: {e}")

Kernel hooked up perfectly! Found tables: [{'Tables_in_project_back': 'products'}, {'Tables_in_project_back': 'reorders'}, {'Tables_in_project_back': 'shipments'}, {'Tables_in_project_back': 'stock_entries'}, {'Tables_in_project_back': 'suppliers'}]


In [2]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install nbformat nbclient

In [4]:
import sys
import os
import pandas as pd
import plotly.express as px

# Establish connection context
db = connect_to_db()
cursor = db.cursor(dictionary=True)

# 1. Define the advanced SQL Window analysis query
abc_query = """
WITH ProductRevenue AS (
    SELECT 
        p.product_id,
        p.product_name,
        SUM(ABS(se.change_quantity) * p.price) AS revenue
    FROM stock_entries se
    JOIN products p ON se.product_id = p.product_id
    WHERE se.change_type = 'Sale'
    GROUP BY p.product_id, p.product_name
),
CalculatedPercentages AS (
    SELECT 
        product_name,
        revenue,
        SUM(revenue) OVER() AS total_revenue,
        SUM(revenue) OVER(ORDER BY revenue DESC) / SUM(revenue) OVER() * 100 AS cumulative_percentage
    FROM ProductRevenue
)
SELECT 
    product_name,
    ROUND(revenue, 2) as revenue,
    CASE 
        WHEN cumulative_percentage <= 80 THEN 'A (High Value)'
        WHEN cumulative_percentage <= 95 THEN 'B (Medium Value)'
        ELSE 'C (Low Value)'
    END AS abc_class
FROM CalculatedPercentages;
"""

# 2. Run the query and load it cleanly into a Pandas DataFrame
cursor.execute(abc_query)
abc_records = cursor.fetchall()
df_abc = pd.DataFrame(abc_records)

# 3. Display the raw analytics summary
print("--- INVENTORY ABC ANALYSIS PREVIEW ---")
print(df_abc.head(10))

# 4. Generate an interactive financial chart to test visual distribution
fig = px.pie(
    df_abc, 
    names='abc_class', 
    values='revenue', 
    hole=0.4,
    title="Revenue Distribution by Inventory Stratification Class",
    color_discrete_sequence=px.colors.sequential.Plotly3
)
fig.show()

cursor.close()
db.close()

--- INVENTORY ABC ANALYSIS PREVIEW ---
      product_name    revenue       abc_class
0        Ask Table  789196.32  A (High Value)
1   Audience Snack  710449.68  A (High Value)
2        Floor Toy  636133.68  A (High Value)
3    Ground Device  623426.22  A (High Value)
4       Whom Table  622170.42  A (High Value)
5       Poor Snack  574728.96  A (High Value)
6      Feeling Toy  562362.60  A (High Value)
7      Sound Snack  547651.80  A (High Value)
8  Suddenly Device  541467.72  A (High Value)
9    Mission Snack  537419.94  A (High Value)
